# 03 — Labeling & Splitting

**Goal:** Load the cleaned labeled data from the previous step, verify label distribution, encode category names as integers, perform a stratified train/val/test split, and save all outputs to the shared Google Drive.

**Depends on:** `data/processed/complaints_labeled_clean.csv` produced by `02_text_cleaning.ipynb`

**Outputs (saved to Drive):**
- `NLP-Complaints-Team/data/processed/train.csv`
- `NLP-Complaints-Team/data/processed/val.csv`
- `NLP-Complaints-Team/data/processed/test.csv`
- `NLP-Complaints-Team/data/processed/label_map.json`

> **Before running:** make sure `02_text_cleaning.ipynb` has been run and its CSVs are present in the shared Drive folder.

In [ ]:
# Cell 2 — Mount Drive + pull repo
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/repo'
os.system(f'git -C "{REPO_DIR}" pull')
print('Repo updated.')

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys

packages = [
    'scikit-learn>=1.3',
    'camel-tools>=1.5',
    'pandas>=2.0',
    'numpy>=1.24',
    'joblib>=1.3',
    'fastapi>=0.110',
    'uvicorn>=0.29',
    'gradio>=4.0',
    'seaborn>=0.13',
    'matplotlib>=3.8',
]

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages
)
print('All packages installed.')

In [ ]:
# Cell 4 — Load cleaned labeled data
import pandas as pd

PROC_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/data/processed'

df = pd.read_csv(f'{PROC_DIR}/complaints_labeled_clean.csv')
print(f'Total rows: {len(df)}')
print('\nLabel distribution:')
print(df['source_label'].value_counts())

In [ ]:
# Cell 5 — Encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['source_label'])

print('Label mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} \u2192 {cls}')

In [ ]:
# Cell 6 — Stratified train/val/test split (70 / 15 / 15)
from sklearn.model_selection import train_test_split

X = df[['text_clean']]
y = df['label_id']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

In [ ]:
# Cell 7 — Save splits
import os

OUT_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

for split_name, X_split, y_split in [
    ('train', X_train, y_train),
    ('val',   X_val,   y_val),
    ('test',  X_test,  y_test),
]:
    split_df = X_split.copy()
    split_df['label_id'] = y_split.values
    split_df.to_csv(f'{OUT_DIR}/{split_name}.csv', index=False)
    print(f'Saved {split_name}.csv ({len(split_df)} rows)')

# Save label mapping for use in model training
import json
mapping = {int(i): cls for i, cls in enumerate(le.classes_)}
with open(f'{OUT_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print('Saved label_map.json')

In [ ]:
# Cell 8 — Verify splits: class distribution per split
for split_name in ['train', 'val', 'test']:
    split_df = pd.read_csv(f'{OUT_DIR}/{split_name}.csv')
    print(f'\n--- {split_name} ({len(split_df)} rows) ---')
    print(split_df['label_id'].value_counts().sort_index())

## Cell 9 — Commit notebook to git

After verifying the outputs above, commit and push **only the notebook** on your branch:

```bash
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo add notebooks/03_labeling_splitting.ipynb
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo commit -m "Add labeling and splitting notebook"
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo push origin khowla/data-processing
```

> **Do NOT commit the CSV or JSON files** — they live only on the shared Drive.